<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 1 (AI): GenAI Foundations — Calling LLMs, Structured Output & Prompting

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Place GenAI** inside AI / ML / Deep Learning
2. **Call an LLM** with the OpenAI client — and reach **Gemini** with the same code
3. **Use LiteLLM** to swap providers in one line
4. **Read token usage** and get **structured (Pydantic) output**
5. **Write better prompts** and recognize **how GenAI fails**

---

## 1. Environment Setup

Run these cells first. You'll need an **OpenAI API key**; Gemini is optional (press Enter to skip).

In [ ]:
# Install the packages we need
!pip install -q openai litellm pydantic

In [ ]:
# Imports
import os
import json
from getpass import getpass
from openai import OpenAI
from pydantic import BaseModel, ValidationError

In [ ]:
# API keys (typed securely — not shown on screen)
openai_api_key = getpass("Enter your OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = openai_api_key

# Gemini is optional — press Enter to skip
google_api_key = getpass("Enter your Gemini API Key (or press Enter to skip): ")

# Models we'll use (change to any you have access to)
OPENAI_MODEL = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.0-flash"

print("Keys set. OpenAI model:", OPENAI_MODEL)

## 2. AI vs ML vs Generative AI

**AI ⊃ ML ⊃ Deep Learning ⊃ GenAI**

| Layer | Meaning | Example |
|---|---|---|
| **AI** | any "smart" task | Google Maps routing |
| **ML** | learns patterns from data | a spam filter |
| **Deep Learning** | ML with neural networks | face unlock, translation |
| **Generative AI** | creates **new content** | ChatGPT writing an email |

**The two classic ML jobs**
- **Regression → a number:** house price · exam score · tomorrow's temperature
- **Classification → a category:** spam/not-spam · benign/malignant · loan default yes/no

**Discriminative vs Generative:** classic ML *draws a boundary* (label/number); GenAI *models the data* and produces **new content**.

## 3. Your First LLM Call (OpenAI)

The **Chat Completions API** takes a list of `messages` and returns the model's reply.

In [ ]:
# Call OpenAI with the official client
client = OpenAI()  # reads OPENAI_API_KEY from the environment

resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise CS tutor."},
        {"role": "user", "content": "Explain what an API is in 2 sentences."},
    ],
)
print(resp.choices[0].message.content)

### Temperature — control the randomness

`temperature=0` → focused & repeatable. Higher → more creative & varied. **Change the value below and re-run to compare (try `0`, then `1.2`).**

In [ ]:
# Change `temperature` and re-run this cell to compare (try 0, then 1.2)
temperature = 0.7

r = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "Give me a tagline for a coffee shop."}],
    temperature=temperature,
)
print(r.choices[0].message.content)

## 4. Same Client → Gemini

Google exposes an **OpenAI-compatible endpoint**, so the *same* OpenAI client works — only **3 things change**: `api_key`, `base_url`, `model`.

In [ ]:
# Reach Gemini using the SAME OpenAI client
if google_api_key:
    gemini_client = OpenAI(
        api_key=google_api_key,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )
    r = gemini_client.chat.completions.create(
        model=GEMINI_MODEL,
        messages=[{"role": "user", "content": "Explain what an API is in 2 sentences."}],
    )
    print("Gemini says:\n", r.choices[0].message.content)
else:
    print("No Gemini key — skipping. (OpenAI cells still work fine.)")

## 5. One Interface for All — LiteLLM

**LiteLLM** speaks the OpenAI format to 100+ providers. Change the **model string** to switch provider — no new SDK, no rewrite.

In [ ]:
# LiteLLM: change the `model` string to switch provider — same function, same format.
from litellm import completion

if google_api_key:
    os.environ["GEMINI_API_KEY"] = google_api_key

response = completion(
    model="openai/gpt-4o-mini",       # try: "gemini/gemini-2.0-flash"  (needs your Gemini key)
    messages=[{"role": "user", "content": "Say hello in 5 words."}],
)
print(response.choices[0].message.content)

## 6. Tokens & Cost

Models read/write **tokens** (~¾ of a word). You **pay per token**, and context is measured in tokens. Every response reports its usage.

In [ ]:
# See how many tokens a call used
r = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "List 3 uses of Python, one line each."}],
)
print(r.choices[0].message.content)
print("\n--- token usage ---")
print("prompt tokens:    ", r.usage.prompt_tokens)
print("completion tokens:", r.usage.completion_tokens)
print("total tokens:     ", r.usage.total_tokens)

## 7. Structured Output with Pydantic

Free text is hard for code to use. Ask for **JSON** and validate it into a **typed object** you can trust.

In [ ]:
# Force the model's answer into a shape we define
class Recipe(BaseModel):
    title: str
    ingredients: list[str]
    minutes: int

resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content":
               "Give a simple pasta recipe as JSON with keys: "
               "title (str), ingredients (list of str), minutes (int)."}],
    response_format={"type": "json_object"},
)

recipe = Recipe.model_validate_json(resp.choices[0].message.content)
print("Title:  ", recipe.title)
print("Minutes:", recipe.minutes)
print("Items:  ", ", ".join(recipe.ingredients))

## 8. Prompt Engineering — Weak vs Strong

A good prompt = **Role + Task + Context + Format**. Same model, very different results. Run both cells and compare.

In [ ]:
# Weak prompt — vague, no guidance
weak = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "tell me about sorting"}],
)
print(weak.choices[0].message.content)

In [ ]:
# Strong prompt — Role + Task + Format
strong = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content":
               "You are a DSA tutor. Explain bubble sort to a 2nd-year student "
               "in exactly 3 bullet points, then give its time complexity."}],
)
print(strong.choices[0].message.content)

## 9. When GenAI Fails (real cases)

| Failure | Real case | Lesson |
|---|---|---|
| **Hallucination** | Air Canada's bot invented a refund policy → airline had to pay (2024) | Confident ≠ correct; verify |
| **Bias** | Gemini image-gen drew historically wrong "diverse" images → paused (2024) | Test across groups |
| **Prompt injection** | Chevy bot talked into a "$1, legally binding" car (2023) | Don't trust user input blindly |
| **Security** | "nullifAI" malicious models slipped onto Hugging Face (2025) | Models are code — scan/pin sources |
| **Misuse** | $25M deepfake video-call fraud at Arup (2024) | Verify identity; disclose AI |

**What to do:** verify outputs · test for bias · limit bot authority · scan model sources · keep a human in the loop.

## 10. Exercises

Q1-Q3: fill in the blanks (`___`). Q4-Q5: write it yourself. Run each cell.

### Q1: Your own OpenAI call

Ask the model **"Give me 3 tips to prepare for a coding interview"** and print the answer.

**Hints:** use `client.chat.completions.create`, set `model=OPENAI_MODEL`, one `user` message, and read the `.content`.

In [ ]:
r = client.chat.completions.create(
    model=___,                       # which model constant?
    messages=[{"role": "___", "content": "___"}],
)
print(r.choices[0].message.___)      # which attribute holds the text?

### Q2: Switch providers with LiteLLM

Send **"Name one famous scientist and their field"** through LiteLLM using **`openai/gpt-4o-mini`**.

**Hints:** call `completion(model=..., messages=[...])`.

In [ ]:
from litellm import completion

r = completion(
    model="___",                     # provider/model, e.g. openai/gpt-4o-mini
    messages=[{"role": "user", "content": "___"}],
)
print(r.choices[0].message.content)

### Q3: Structured output

Get one **book** as JSON with keys `title`, `author`, `year`, and validate it with the `Book` model.

**Hints:** add `response_format={"type": "json_object"}`; parse with `Book.model_validate_json(...)`.

In [ ]:
class Book(BaseModel):
    title: str
    author: str
    year: int

resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content":
               "Give one classic novel as JSON with keys: title, author, year (int)."}],
    response_format={"type": "___"},          # what format?
)

book = Book.model_validate_json(resp.choices[0].message.___)
print(book.title, "by", book.author, "(", book.year, ")")

### Q4: Tagline generator

Use a **system prompt** to make the model a creative branding expert, then ask it for a **one-line tagline** for a company of your choice. Set a **high temperature** and run it a few times — the tagline should change each run.

In [ ]:
# your code here


### Q5: Two-provider tagline race

Using **LiteLLM**, send the same tagline prompt (for a company you pick) through **`openai/gpt-4o-mini`** and **`gemini/gemini-2.0-flash`**, and print both answers to compare.

In [ ]:
# your code here


---

### ✅ Recap

- LLMs generate content by predicting tokens; you steer them with **prompts**.
- One **OpenAI client** reaches OpenAI **and** Gemini; **LiteLLM** reaches everything.
- Ask for **JSON** + validate with **Pydantic** to make output code-safe.
- GenAI **hallucinates and can be misused** — build with guardrails.

---

## 🎁 Bonus (optional): Pydantic *validation*

Pydantic doesn't just shape data — it **checks it**. Give each field a type and simple rules; bad data is rejected with a clear error. *(This is exactly what FastAPI uses to validate API requests in Week 2.)*

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class Student(BaseModel):
    name: str = Field(min_length=1)     # can't be empty
    age: int = Field(ge=15, le=100)     # must be 15-100
    interests: list[str] = []           # optional, defaults to empty

good = Student(name="Ada", age=20, interests=["ai", "math"])
print(good)

In [ ]:
# Pydantic rejects invalid data and tells you exactly what's wrong
try:
    Student(name="", age=200)           # empty name + age too high
except ValidationError as e:
    print(e)

### Bonus exercise

Add a rule so a **negative** `age` is rejected, and cap `name` at a **maximum** length of 50. Fill in the blanks (`___`) and run.

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class Person(BaseModel):
    name: str = Field(min_length=1, max_length=___)   # at most 50 characters
    age: int = Field(ge=___)                          # 0 or more (no negatives)

print(Person(name="Ravi", age=19))       # should work

try:
    Person(name="Ravi", age=-5)          # should be caught
except ValidationError as e:
    print(e)